# FAQ Next-Word Predictor with Dropout and Early Stopping

An improved version of the FAQ next-word model: it adds dropout, a validation split, `EarlyStopping` and `ModelCheckpoint` so that the best model is saved automatically. It then completes questions typed by the user.

**How to run:** Open in Google Colab and choose **Runtime > Run all** (a CPU runtime is fine).

_Rewritten from my original Colab notebook: `chjatgpt simple and better faq.ipynb`._

## 1. Imports and seed

In [ ]:
import re
import urllib.request

import numpy as np
import tensorflow as tf
import keras
from keras import layers

SEED = 42
keras.utils.set_random_seed(SEED)  # seeds Python, NumPy and TensorFlow
print("TensorFlow:", tf.__version__)

## 2. The data
A small, made-up FAQ text written inline so the notebook runs on its own.

In [ ]:
FAQ_TEXT = """What is the course fee for the data science program?
The course follows a monthly subscription model and you pay a small fee every month.
What is the total duration of the course?
The total duration of the course is seven months.
What is the syllabus of the program?
We cover python fundamentals, python libraries for data science, data analysis, sql, maths for machine learning, machine learning algorithms and case studies.
Will deep learning and nlp be a part of this program?
No, deep learning and nlp are not a part of this program.
What if I miss a live session?
All sessions are recorded, so you can watch the recording later.
Where can I find the class schedule?
The class schedule is shared on the course dashboard every month.
What is the time duration of the live sessions?
Most live sessions last about two hours.
How will I be informed about the upcoming class?
You will get an email reminder before every upcoming class.
Can I do this course if I am from a non technical background?
Yes, the course starts from the basics, so anyone can join.
I joined the program late. Can I still pay the fee for earlier months?
Yes, you can pay for the earlier months and watch all the recorded sessions.
Where do I have to submit the tasks?
You do not need to submit tasks, but solutions are shared for practice.
Will we do case studies in the program?
Yes, we will do many case studies on real datasets.
Where can I contact the support team?
You can contact the support team from the help section of the course website.
Is there any certificate after completing the course?
Yes, you get a certificate after you finish the course and the final test.
"""

## 3. Tokenize
Each word gets an integer id (0 is kept for padding).

In [ ]:
def clean_words(line):
    """Lower-case a line and keep only words (letters and apostrophes)."""
    line = line.lower().replace("\u2019", "'")
    return re.findall(r"[a-z']+", line)


def build_vocab(lines):
    """Map every word to an integer id. Id 0 is reserved for padding."""
    words = sorted({w for line in lines for w in clean_words(line)})
    word_to_id = {w: i + 1 for i, w in enumerate(words)}
    id_to_word = {i: w for w, i in word_to_id.items()}
    return word_to_id, id_to_word


def encode(line, word_to_id):
    return [word_to_id[w] for w in clean_words(line) if w in word_to_id]

In [ ]:
lines = [l for l in FAQ_TEXT.split("\n") if l.strip()]
word_to_id, id_to_word = build_vocab(lines)
vocab_size = len(word_to_id) + 1
print("Vocabulary size:", vocab_size)
print(list(word_to_id.items())[:10])

## 4. Build input sequences
For the line `what is the course fee` we create `what -> is`, `what is -> the`, ... The last word is the label.

In [ ]:
def make_ngram_dataset(lines, word_to_id, max_len=None):
    """For each line 'a b c d' create the samples [a]->b, [a b]->c, [a b c]->d."""
    sequences = []
    for line in lines:
        ids = encode(line, word_to_id)
        for i in range(1, len(ids)):
            sequences.append(ids[: i + 1])
    if max_len is None:
        max_len = max(len(s) for s in sequences)
    padded = keras.utils.pad_sequences(sequences, maxlen=max_len, padding="pre", truncating="pre")
    X, y = padded[:, :-1], padded[:, -1]
    return X, y, max_len

In [ ]:
X, y, max_len = make_ngram_dataset(lines, word_to_id)
print("X:", X.shape, "| y:", y.shape, "| max length:", max_len)
print("Example input:", X[3], "-> label:", y[3])

## 5. Model with dropout

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(max_len - 1,)),
    layers.Embedding(vocab_size, 100),
    layers.LSTM(150, return_sequences=True),
    layers.Dropout(0.2),
    layers.LSTM(150),
    layers.Dropout(0.2),
    layers.Dense(vocab_size, activation="softmax"),
])
model.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
model.summary()

## 6. Train with callbacks
We shuffle the samples first so the validation split is not only the last FAQ lines.

In [ ]:
order = np.random.default_rng(SEED).permutation(len(X))
X, y = X[order], y[order]

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint("best_faq_model.keras", monitor="val_loss", save_best_only=True),
]
history = model.fit(X, y, epochs=150, validation_split=0.15, callbacks=callbacks, verbose=2)

## 7. Evaluate

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history["loss"], label="train loss")
if "val_loss" in history.history:
    plt.plot(history.history["val_loss"], label="validation loss")
plt.xlabel("epoch")
plt.legend()
plt.show()

In [ ]:
val_count = int(len(X) * 0.15)
loss, acc = model.evaluate(X[-val_count:], y[-val_count:], verbose=0)
print(f"Validation loss: {loss:.3f} | accuracy: {acc:.3f}")

## 8. Predict

In [ ]:
def predict_next_word(model, seed_text, word_to_id, id_to_word, max_len):
    ids = encode(seed_text, word_to_id)
    padded = keras.utils.pad_sequences([ids], maxlen=max_len - 1, padding="pre", truncating="pre")
    probs = model.predict(padded, verbose=0)[0]
    probs[0] = 0  # never predict the padding id
    return id_to_word.get(int(np.argmax(probs)), "")


def generate_text(model, seed_text, n_words, word_to_id, id_to_word, max_len):
    text = seed_text
    for _ in range(n_words):
        next_word = predict_next_word(model, text, word_to_id, id_to_word, max_len)
        if not next_word:
            break
        text += " " + next_word
    return text

In [ ]:
for seed in ["what is the total", "where can i", "is there any"]:
    print(generate_text(model, seed, 8, word_to_id, id_to_word, max_len))